In [ ]:
import os
import json
from tqdm import tqdm
from llama_cpp import Llama

# Configuration
MODEL_PATH = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/Orion_14B_q8.gguf"
INPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/TiJian94.json"
OUTPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/results_with_orion.json"

GPU_LAYERS = -1
N_CTX = 32768               
N_BATCH = 1024
N_UBATCH = 512
N_THREADS = 4
MAIN_GPU = 0
MAX_TOKENS = 4096
TEMPERATURE = 0.1
TOP_P = 0.8
SAVE_EVERY = 10

# Load model
print("Loading Orion GGUF...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=GPU_LAYERS, n_ctx=N_CTX, n_batch=N_BATCH, n_ubatch=N_UBATCH,
    n_threads=N_THREADS, verbose=False,
)
print("Model Loaded!")

# Read data
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    dataset = json.load(f)
print(f"Total records: {len(dataset)}")

# Resume from checkpoint
results = []
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        results = json.load(f)
    start_idx = len(results)
    print(f"Resuming from record {start_idx}")
else:
    start_idx = 0

# Inference
for idx in tqdm(range(start_idx, len(dataset)), desc="Inference"):
    item = dataset[idx]
    instruction = item.get("instruction", "")
    input_text = item.get("input", "")
    gt_output = item.get("output", "")

    try:
        # Official Orion Chat Prompt
        prompt = f"""<|im_start|>user
{instruction}

{input_text}
<|im_end|>
<|im_start|>assistant
"""
        output = llm(
            prompt, max_tokens=MAX_TOKENS, temperature=TEMPERATURE, 
            top_p=TOP_P, stop=["<|im_end|>"]
        )
        answer = output["choices"][0]["text"].strip()
        results.append({
            "instruction": instruction, "input": input_text, 
            "output": gt_output, "output_orion": answer
        })
    except Exception as e:
        print(f"\nFailed on record {idx}: {e}")
        results.append({
            "instruction": instruction, "input": input_text, 
            "output": gt_output, "output_orion": "", "error": str(e)
        })

    # Save every N records
    if len(results) % SAVE_EVERY == 0:
        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=4)
        print(f"Saved {len(results)} records")

# Final save
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("All completed!")

Loading Orion GGUF...


llama_context: n_ctx_seq (32768) > n_ctx_train (4096) -- possible training context overflow


Model Loaded!
Total records: 94


Inference:  11%|█         | 10/94 [08:59<53:03, 37.89s/it]  

已保存 10 条


Inference:  21%|██▏       | 20/94 [12:41<32:09, 26.07s/it]

... (13 outputs left out)

Inference:  96%|█████████▌| 90/94 [48:49<01:22, 20.55s/it]

已保存 90 条


Inference: 100%|██████████| 94/94 [50:39<00:00, 32.33s/it]

全部完成！
